# Kaggle Full Training - Plant Classification

Notebook nay dung de train day du 3 model tren dataset plant classification:
- Custom CNN
- ResNet50 Feature Extraction
- ResNet50 Fine-tuning

Output duoc luu vao `/kaggle/working/plant_training_outputs`.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/plant_training_outputs')

def find_dataset_dir(dataset_name='dataset_plant_classification'):
    direct_candidates = [
        INPUT_ROOT / 'plant_classification' / dataset_name,
        INPUT_ROOT / dataset_name,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    for candidate in sorted(INPUT_ROOT.rglob(dataset_name)):
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError(f'Could not find {dataset_name} under {INPUT_ROOT}')

DATASET_DIR = find_dataset_dir()

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 50
LEARNING_RATE_CNN = 1e-3
LEARNING_RATE_FE = 1e-3
LEARNING_RATE_FT = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 7
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42
USE_AMP = True
SAVE_EVERY_EPOCH = True
USE_PRETRAINED_WEIGHTS = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8
print('DATASET_DIR =', DATASET_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
import copy
import json
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png'}

class_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

samples = []
class_counts = {}

for class_dir in class_dirs:
    images = sorted([
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in VALID_EXTS
    ])
    class_counts[class_dir.name] = len(images)
    for img_path in images:
        samples.append((img_path, class_to_idx[class_dir.name]))

print('Total classes:', len(class_names))
print('Total images:', len(samples))
print('Min class size:', min(class_counts.values()))
print('Max class size:', max(class_counts.values()))
pd.Series(class_counts).sort_values(ascending=False).head(10)

In [ ]:
def stratified_split(samples, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    by_class = {}
    for img_path, label in samples:
        by_class.setdefault(label, []).append((img_path, label))

    train_samples = []
    val_samples = []
    test_samples = []

    for label, items in by_class.items():
        items = items.copy()
        rng.shuffle(items)
        n = len(items)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        if train_end < 1:
            train_end = 1
        if val_end <= train_end:
            val_end = min(train_end + 1, n)
        if val_end >= n:
            val_end = n - 1

        train_samples.extend(items[:train_end])
        val_samples.extend(items[train_end:val_end])
        test_samples.extend(items[val_end:])

    return train_samples, val_samples, test_samples

train_samples, val_samples, test_samples = stratified_split(samples, TRAIN_RATIO, VAL_RATIO, SEED)

print('Train:', len(train_samples))
print('Val:', len(val_samples))
print('Test:', len(test_samples))

In [ ]:
class ImageListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label = self.samples[index]
        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageListDataset(train_samples, transform=train_transform)
val_dataset = ImageListDataset(val_samples, transform=eval_transform)
test_dataset = ImageListDataset(test_samples, transform=eval_transform)

pin_memory = DEVICE.type == 'cuda'

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def load_resnet50_backbone():
    if not USE_PRETRAINED_WEIGHTS:
        print('USE_PRETRAINED_WEIGHTS=False -> using ResNet50 without pretrained weights')
        return models.resnet50(weights=None)

    try:
        return models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    except Exception as exc:
        print(f'Warning: could not load pretrained ResNet50 weights: {exc}')
        print('Falling back to weights=None so the notebook can continue offline.')
        return models.resnet50(weights=None)

def build_resnet50_feature_extractor(num_classes):
    model = load_resnet50_backbone()
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def build_resnet50_finetune(num_classes):
    model = load_resnet50_backbone()
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast(device_type='cuda', enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits, labels)

            if is_train:
                if scaler is not None and AMP_ENABLED:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision_macro': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall_macro': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
    }
    return metrics, all_labels, all_preds

def evaluate_model(model, loader, criterion):
    return run_epoch(model, loader, criterion, optimizer=None, scaler=None)


In [ ]:
def save_confusion_matrix(labels, preds, class_names, output_path, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(18, 15))
    sns.heatmap(cm, cmap='Blues', cbar=True)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

def save_history_plot(history_df, output_path, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
    axes[0].set_title(f'{model_name} Loss')
    axes[0].legend()
    axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train')
    axes[1].plot(history_df['epoch'], history_df['val_accuracy'], label='val')
    axes[1].set_title(f'{model_name} Accuracy')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


In [ ]:
def train_model(model_name, model, learning_rate, epochs=50):
    model_dir = OUTPUT_ROOT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda') if AMP_ENABLED else None

    history = []
    best_val_acc = -1.0
    best_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
        val_metrics, _, _ = evaluate_model(model, val_loader, criterion)

        row = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'train_f1_macro': train_metrics['f1_macro'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_f1_macro': val_metrics['f1_macro'],
        }
        history.append(row)
        print(f"[{model_name}] Epoch {epoch:02d} | train_acc={row['train_accuracy']:.4f} | val_acc={row['val_accuracy']:.4f} | val_loss={row['val_loss']:.4f}")

        if SAVE_EVERY_EPOCH:
            torch.save(model.state_dict(), model_dir / f'epoch_{epoch:02d}.pth')

        if row['val_accuracy'] > best_val_acc:
            best_val_acc = row['val_accuracy']
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, model_dir / 'best_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f'[{model_name}] Early stopping triggered.')
            break

    elapsed = time.time() - start_time
    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics, test_labels, test_preds = evaluate_model(model, test_loader, criterion)
    history_df = pd.DataFrame(history)
    history_df.to_csv(model_dir / 'history.csv', index=False)
    save_history_plot(history_df, model_dir / 'history.png', model_name)
    save_confusion_matrix(test_labels, test_preds, class_names, model_dir / 'confusion_matrix.png', f'{model_name} Test Confusion Matrix')

    summary = {
        'model_name': model_name,
        'epochs_completed': len(history),
        'best_val_accuracy': best_val_acc,
        'test_metrics': test_metrics,
        'runtime_seconds': elapsed,
        'num_classes': len(class_names),
        'train_samples': len(train_samples),
        'val_samples': len(val_samples),
        'test_samples': len(test_samples),
    }

    with open(model_dir / 'summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    return summary, history_df

In [ ]:
training_plan = [
    ('custom_cnn', CustomCNN(num_classes=len(class_names)), LEARNING_RATE_CNN),
    ('resnet50_feature_extraction', build_resnet50_feature_extractor(num_classes=len(class_names)), LEARNING_RATE_FE),
    ('resnet50_fine_tuning', build_resnet50_finetune(num_classes=len(class_names)), LEARNING_RATE_FT),
]

training_plan

In [ ]:
all_summaries = []

for model_name, model, lr in training_plan:
    print(f'\n===== Training {model_name} =====')
    summary, history_df = train_model(model_name, model, lr, epochs=EPOCHS)
    all_summaries.append(summary)

summary_df = pd.DataFrame([
    {
        'model_name': item['model_name'],
        'epochs_completed': item['epochs_completed'],
        'best_val_accuracy': item['best_val_accuracy'],
        'test_accuracy': item['test_metrics']['accuracy'],
        'test_f1_macro': item['test_metrics']['f1_macro'],
        'test_precision_macro': item['test_metrics']['precision_macro'],
        'test_recall_macro': item['test_metrics']['recall_macro'],
        'runtime_minutes': item['runtime_seconds'] / 60.0,
    }
    for item in all_summaries
])

summary_df.to_csv(OUTPUT_ROOT / 'all_models_summary.csv', index=False)
with open(OUTPUT_ROOT / 'all_models_summary.json', 'w', encoding='utf-8') as f:
    json.dump(all_summaries, f, indent=2)

summary_df

In [ ]:
summary_df.plot(x='model_name', y=['test_accuracy', 'test_f1_macro'], kind='bar', figsize=(10, 5), rot=20)
plt.title('Model Comparison on Test Set')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'model_comparison.png', dpi=200)
plt.show()

In [ ]:
print('Saved output files:')
for path in sorted(OUTPUT_ROOT.iterdir()):
    print('-', path.name)